In [1]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [3]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random
import pickle

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from tensorflow import keras
from tensorflow.keras import layers, optimizers, models
from scipy.signal import find_peaks

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [5]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [6]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [7]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [9]:
two_interval_minutes = 2

nf_index_symbol, nf_quantity = get_index_symbol_and_quantity("Nifty")
bnf_index_symbol, bnf_quantity = get_index_symbol_and_quantity("Bank Nifty")

#nf_train_df = fetch_candle_data(100, nf_index_symbol, two_interval_minutes)
#bnf_train_df = fetch_candle_data(100, bnf_index_symbol, two_interval_minutes)

nf_train_df = fetch_train_candle_data(5, nf_index_symbol, two_interval_minutes)
bnf_train_df = fetch_train_candle_data(5, bnf_index_symbol, two_interval_minutes)

print(len(nf_train_df), len(bnf_train_df))

nf_train_df = nf_train_df.drop_duplicates(subset='datetime', keep='first')
bnf_train_df = bnf_train_df.drop_duplicates(subset='datetime', keep='first')

print(len(nf_train_df), len(bnf_train_df))

64434 64434
63870 63870


In [10]:
class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame, base_interval_minutes: int):
        """
        df: DataFrame with columns [datetime, open, high, low, close, volume (optional)]
            'datetime' is a Unix timestamp in seconds.
        base_interval_minutes: The base timeframe in minutes (e.g. 2, 5, 15, etc.).
        """
        self.df = df.copy()
        self.base_interval = base_interval_minutes

        # We no longer need an htf_map since higher timeframe features are removed
        # self.htf_map = { ... }  # Removed

    def preprocess_datetime(self):
        """
        1) Convert Unix timestamp to local time (Asia/Kolkata).
        2) Check for duplicates/missing timestamps.
        3) Sort by time and set 'datetime' as the index.
        """
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        # Check for duplicates/missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        """
        Optionally drop 'volume' column if it contains zeros or NaNs.
        Adjust if you prefer different volume handling.
        """
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        """
        Compute a few standard technical indicators on the base timeframe:
        RSI, MACD, Bollinger Bands, ATR, etc.
        """
        self.df.sort_index(inplace=True)

        # RSI on base timeframe close
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR (for baseline volatility measure)
        self.df['atr_base'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        )

        return self

    def add_candlestick_patterns(self):
        """
        Detect a few common candlestick patterns.
        We store them as numeric columns (0 = not present, 1 = bullish pattern, -1 = bearish pattern, etc.)
        so that the pipeline remains consistent with numeric-only scaling.
        """
        self.df.sort_index(inplace=True)

        # Example: Doji detection (very simplistic)
        # We check if the open and close are nearly the same
        body_size = (self.df['close'] - self.df['open']).abs()
        candle_range = self.df['high'] - self.df['low']
        self.df['candle_doji'] = np.where(
            (body_size <= 0.1 * candle_range), 1.0, 0.0
        )

        # Example: Engulfing pattern (bullish and bearish)
        # - Bullish Engulfing: today's open < yesterday's close, today's close > yesterday's open
        # - Bearish Engulfing: today's open > yesterday's close, today's close < yesterday's open
        # We shift the open/close by 1 day to compare
        self.df['open_shift1'] = self.df['open'].shift(1)
        self.df['close_shift1'] = self.df['close'].shift(1)

        cond_bull_engulf = (self.df['open'] < self.df['close_shift1']) & (self.df['close'] > self.df['open_shift1'])
        cond_bear_engulf = (self.df['open'] > self.df['close_shift1']) & (self.df['close'] < self.df['open_shift1'])

        self.df['candle_engulf'] = np.where(cond_bull_engulf, 1.0,
                                    np.where(cond_bear_engulf, -1.0, 0.0))

        # Drop helper columns to keep dataset clean
        self.df.drop(['open_shift1', 'close_shift1'], axis=1, inplace=True)

        return self

    def add_swing_points(self, left_bars=3, right_bars=3):
        """
        Detect local swing highs and lows.
        left_bars/right_bars define how many bars on each side must be lower/higher (for highs/lows).
        """
        self.df.sort_index(inplace=True)

        # For each row, check if 'high' is greater than the preceding and following N bars
        rolling_high = self.df['high'].rolling(left_bars+1, center=False).max()
        rolling_low = self.df['low'].rolling(left_bars+1, center=False).min()

        # We'll do a simplistic approach: shift rolling computations accordingly
        # A more robust approach might be to iterate row by row, but we'll keep it vectorized for brevity.

        # Swing High
        # Compare current high to future bars as well, so we do a forward rolling
        forward_high = self.df['high'].shift(-right_bars).rolling(right_bars+1, center=False).max()
        self.df['is_swing_high'] = 0.0
        cond_high = (self.df['high'] == rolling_high) & (self.df['high'] == forward_high)
        self.df.loc[cond_high, 'is_swing_high'] = 1.0

        # Swing Low
        forward_low = self.df['low'].shift(-right_bars).rolling(right_bars+1, center=False).min()
        self.df['is_swing_low'] = 0.0
        cond_low = (self.df['low'] == rolling_low) & (self.df['low'] == forward_low)
        self.df.loc[cond_low, 'is_swing_low'] = 1.0

        return self

    def add_range_breakout_features(self, window=20):
        """
        Add Donchian channels, breakout flags, and a simple measure of range expansion.
        """
        self.df.sort_index(inplace=True)

        # Donchian Channels
        self.df['donchian_high'] = self.df['high'].rolling(window).max()
        self.df['donchian_low'] = self.df['low'].rolling(window).min()
        self.df['donchian_range'] = self.df['donchian_high'] - self.df['donchian_low']

        # Breakout flags
        #  1.0 if close > donchian_high, -1.0 if close < donchian_low, else 0.0
        self.df['donchian_breakout'] = np.where(
            self.df['close'] > self.df['donchian_high'], 1.0,
            np.where(self.df['close'] < self.df['donchian_low'], -1.0, 0.0)
        )

        # Range expansion/compression example
        # Rolling std of (high - low)
        self.df['range_expansion'] = (self.df['high'] - self.df['low']).rolling(window).std()

        return self

    def add_momentum_features(self):
        """
        Add additional momentum-based indicators like Stochastics, ADX, CCI, etc.
        (We already have RSI, MACD in add_base_timeframe_features, but you can unify them here if you prefer.)
        """
        self.df.sort_index(inplace=True)

        # Stochastic
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14, d=3
        )
        self.df['stoch_k'] = stoch['STOCHk_14_3_3']
        self.df['stoch_d'] = stoch['STOCHd_14_3_3']

        # ADX (Average Directional Movement Index)
        adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
        self.df['adx'] = adx_data['ADX_14']
        self.df['diplus'] = adx_data['DMP_14']
        self.df['diminus'] = adx_data['DMN_14']

        # CCI (Commodity Channel Index)
        self.df['cci'] = ta.cci(self.df['high'], self.df['low'], self.df['close'], length=20)

        return self

    def add_volatility_features(self, window=20):
        """
        Add additional volatility-based features, e.g. historical volatility, Z-score of returns, etc.
        """
        self.df.sort_index(inplace=True)

        # Historical volatility (using close-to-close log returns)
        self.df['log_return'] = np.log(self.df['close'] / self.df['close'].shift(1))
        self.df['hist_vol'] = self.df['log_return'].rolling(window).std() * np.sqrt(252)  # Annualized approximation

        # Z-Score of Price Changes over 'window'
        rolling_mean = self.df['log_return'].rolling(window).mean()
        rolling_std = self.df['log_return'].rolling(window).std()
        self.df['zscore_return'] = (self.df['log_return'] - rolling_mean) / (rolling_std + 1e-9)

        return self

    def add_volume_features(self):
        """
        Add volume-based features if 'volume' is present.
        """
        if 'volume' not in self.df.columns:
            return self  # No volume data, skip

        self.df.sort_index(inplace=True)

        # On-Balance Volume
        self.df['obv'] = ta.obv(self.df['close'], self.df['volume'])

        # Volume spike detection:
        # Compare current volume to rolling average
        vol_mean = self.df['volume'].rolling(20).mean()
        vol_std = self.df['volume'].rolling(20).std()
        self.df['vol_spike'] = np.where(
            (self.df['volume'] > (vol_mean + 2 * vol_std)), 1.0, 0.0
        )

        # VWAP over the day (if you have daily or session logic, you'd groupby date)
        # Here, for a simpler approach, we do a rolling 1-day approximation:
        # This might not be perfectly accurate if you have continuous 24h data, but it’s a start.
        # Usually you’d reset VWAP each day or each session.
        typical_price = (self.df['high'] + self.df['low'] + self.df['close']) / 3.0
        self.df['cum_tp_vol'] = (typical_price * self.df['volume']).cumsum()
        self.df['cum_vol'] = self.df['volume'].cumsum()
        self.df['vwap'] = self.df['cum_tp_vol'] / (self.df['cum_vol'] + 1e-9)

        # Drop helper columns
        self.df.drop(['cum_tp_vol', 'cum_vol'], axis=1, inplace=True)

        return self

    def add_regime_features(self):
        """
        Classify each bar as 'trending' or 'ranging' (0 or 1) based on ADX, or volatility-based thresholds, etc.
        Keep it numeric for scaling (e.g., 0.0 or 1.0).
        """
        self.df.sort_index(inplace=True)

        # If ADX above 25 => trending, else ranging
        # (25 is just a typical reference threshold; pick whichever fits your strategy)
        if 'adx' not in self.df.columns:
            # If ADX wasn't yet computed, do it quickly
            adx_data = ta.adx(self.df['high'], self.df['low'], self.df['close'], length=14)
            self.df['adx'] = adx_data['ADX_14']

        self.df['regime_trend'] = np.where(self.df['adx'] >= 25, 1.0, 0.0)

        # Alternatively, you could have multiple columns for different regime classifications
        return self

    def add_time_features(self):
        """
        Extract cyclical or numeric time features (hour of day, day of week).
        We ensure they are numeric (float).
        """
        self.df.sort_index(inplace=True)

        # Because 'datetime' is now our index, we can access it via self.df.index
        # Create columns for hour, day of week
        self.df['hour'] = self.df.index.hour.astype(float)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(float)

        # Cyclical encoding for hour
        self.df['hour_sin'] = np.sin(2 * np.pi * self.df['hour'] / 24.0)
        self.df['hour_cos'] = np.cos(2 * np.pi * self.df['hour'] / 24.0)

        # Cyclical encoding for day_of_week
        self.df['dow_sin'] = np.sin(2 * np.pi * self.df['day_of_week'] / 7.0)
        self.df['dow_cos'] = np.cos(2 * np.pi * self.df['day_of_week'] / 7.0)

        # You can drop raw hour/day_of_week if you prefer only cyclical features
        # self.df.drop(['hour', 'day_of_week'], axis=1, inplace=True)

        return self

    def add_adaptive_targets_and_stops(self):
        """
        Instead of fixed (2 * ATR) / (1 * ATR), adapt based on current volatility or regime.
        For demonstration, we’ll say:
            - If regime is trending, target = 3 * ATR, stop = 1.5 * ATR
            - If regime is ranging, target = 1.5 * ATR, stop = 1 * ATR
        This is just an example logic.
        """
        self.df.sort_index(inplace=True)

        if 'atr_base' not in self.df.columns:
            self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        # If we haven't computed 'regime_trend', do so
        if 'regime_trend' not in self.df.columns:
            self.add_regime_features()

        # Example adaptation
        is_trend = (self.df['regime_trend'] == 1.0)
        self.df['Target'] = np.where(is_trend, 3.0 * self.df['atr_base'], 1.5 * self.df['atr_base'])
        self.df['StopLoss'] = np.where(is_trend, 1.5 * self.df['atr_base'], 1.0 * self.df['atr_base'])

        return self

    def run_pipeline(self):
        """
        Run all steps except higher timeframe computations (now removed).
        You can rearrange the order as desired.
        """
        (self.preprocess_datetime()
             .clean_data()
             .add_base_timeframe_features()
             .add_candlestick_patterns()
             .add_swing_points()
             .add_range_breakout_features()
             .add_momentum_features()
             .add_volatility_features()
             .add_volume_features()            # only applies if 'volume' exists
             .add_regime_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
        )
        return self

    def get_processed_df(self):
        """
        Retrieve the final DataFrame (including any signals if label_signals() was called).
        We'll also ensure that all columns except 'Signal' are float with 2 decimal places.
        """
        # Drop rows that have NaN from lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)

        # Drop 'Entry Price' and 'Exit Price' if you do not want them in final features
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        # Ensure 'Signal' is int; everything else float with 2 decimals
        if 'Signal' in self.df.columns:
            sig_col = self.df['Signal'].astype(int)
            self.df.drop(columns=['Signal'], inplace=True)
        else:
            sig_col = None

        # Convert all remaining columns to float
        for col in self.df.columns:
            self.df[col] = self.df[col].astype(float)

        # Round to 2 decimals
        self.df = self.df.round(2)

        # Reinsert Signal column if it exists
        if sig_col is not None:
            self.df['Signal'] = sig_col

        return self.df

nf_train_pipeline = FullFeaturePipeline(nf_train_df, two_interval_minutes)
nf_train_pipeline.run_pipeline()

nf_processed_train_df = nf_train_pipeline.get_processed_df()

bnf_train_pipeline = FullFeaturePipeline(bnf_train_df, two_interval_minutes)
bnf_train_pipeline.run_pipeline()

bnf_processed_train_df = bnf_train_pipeline.get_processed_df()

In [11]:
nf_processed_train_df.columns

Index(['open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
       'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'candle_doji',
       'candle_engulf', 'is_swing_high', 'is_swing_low', 'donchian_high',
       'donchian_low', 'donchian_range', 'donchian_breakout',
       'range_expansion', 'stoch_k', 'stoch_d', 'adx', 'diplus', 'diminus',
       'cci', 'log_return', 'hist_vol', 'zscore_return', 'regime_trend',
       'hour', 'day_of_week', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
       'Target', 'StopLoss'],
      dtype='object')

In [12]:
nf_processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-06 10:21:00,18576.20,18577.80,18573.70,18575.00,35.60,-7.67,0.34,-8.01,18564.86,18586.11,...,0.14,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,19.48,9.74
2023-06-06 10:23:00,18574.80,18576.30,18573.70,18575.30,36.09,-7.52,0.39,-7.92,18565.75,18584.36,...,0.51,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,18.57,9.29
2023-06-06 10:25:00,18575.90,18577.60,18574.10,18576.70,38.44,-7.21,0.56,-7.77,18567.47,18582.78,...,0.74,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,17.95,8.98
2023-06-06 10:27:00,18575.90,18576.40,18565.30,18566.10,29.58,-7.73,0.03,-7.77,18566.41,18581.06,...,-2.06,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,19.20,9.60
2023-06-06 10:29:00,18566.40,18568.80,18565.70,18566.80,30.71,-7.99,-0.18,-7.81,18565.96,18579.46,...,0.53,1.0,10.0,1.0,0.50,-0.87,0.78,0.62,18.44,9.22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-18 15:21:00,24843.65,24856.95,24843.40,24855.75,48.83,-5.78,-1.40,-4.38,24833.63,24858.61,...,1.88,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.90,18.50,12.33
2024-10-18 15:23:00,24857.10,24876.40,24855.65,24875.15,59.75,-3.46,0.74,-4.20,24833.28,24859.24,...,2.43,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.90,19.40,12.93
2024-10-18 15:25:00,24874.55,24876.35,24865.80,24866.40,54.14,-2.30,1.52,-3.82,24833.40,24859.54,...,-1.13,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.90,19.14,12.76


In [13]:
bnf_processed_train_df

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,zscore_return,regime_trend,hour,day_of_week,hour_sin,hour_cos,dow_sin,dow_cos,Target,StopLoss
datetime,,,,,,,,,,,,,,,,,,,,,
2023-06-06 10:21:00,44089.90,44092.10,44082.20,44083.10,39.38,-20.31,3.99,-24.30,44050.78,44101.56,...,-0.20,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,29.95,19.97
2023-06-06 10:23:00,44083.20,44089.00,44079.10,44088.10,42.08,-19.05,4.20,-23.25,44052.67,44098.43,...,0.61,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,28.78,19.18
2023-06-06 10:25:00,44088.60,44093.60,44085.10,44089.70,42.96,-17.72,4.43,-22.14,44057.44,44095.16,...,0.37,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,27.54,18.36
2023-06-06 10:27:00,44087.90,44090.60,44053.10,44054.00,31.49,-19.32,2.26,-21.58,44054.22,44091.04,...,-2.15,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,29.74,19.83
2023-06-06 10:29:00,44057.40,44067.30,44051.90,44060.90,35.10,-19.80,1.42,-21.22,44056.62,44086.91,...,0.75,0.0,10.0,1.0,0.50,-0.87,0.78,0.62,29.24,19.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-18 15:21:00,52111.70,52134.40,52087.25,52129.95,56.14,-3.92,3.12,-7.04,52025.25,52099.85,...,0.79,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.90,61.93,41.29
2024-10-18 15:23:00,52133.55,52153.35,52119.25,52147.35,58.96,0.54,6.07,-5.52,52026.14,52099.36,...,0.67,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.90,61.16,40.78
2024-10-18 15:25:00,52141.95,52155.25,52117.70,52121.10,53.39,1.94,5.97,-4.03,52026.76,52100.58,...,-1.16,0.0,15.0,4.0,-0.71,-0.71,-0.43,-0.90,60.82,40.54


RL Environment

In [14]:
# -----------------------------------------------------------------------------------
# 1) CONFIGURATION CELL
# -----------------------------------------------------------------------------------
# Define all parameters, hyperparameters, data, etc. in a single config dict for easy control.

config = {
    "data_dict": None,                   # { "instrument_name": pd.DataFrame, ... } - user sets externally
    "brokerage_dict": None,              # Add brokerage specific for that instrument.
    "instrument_selection": "random",    # "random", "sequential", or a specific instrument name

    "initial_capital_multiplier": 3.0,   # Env capital = multiplier * max(close) for the chosen instrument
    "flip_position": True,             # if True, Sell on Long will close & flip to Short in one step
    "track_unrealized_in_reward": True, # add incremental reward for open positions
    "unrealized_reward_weight": 0.3,      # New: weight for unrealized PnL (0.3 = 30%)
    "drawdown_penalty_alpha": 15.0,     # penalty scale for drawdown each step => reward -= alpha * dd

    # Reward mode: "raw", "pct", "log", or "clip"
    "reward_mode": "pct",
    "pct_base_capital": "initial",      # "current" or "initial" for 'pct'/'log' modes
    "clip_max_abs_reward": 1e5,         # used if reward_mode="clip"
    "reward_scaling_factor": 1.0,

    "max_drawdown_percent": 0.5,        # end episode if drawdown >= 50% (None to disable)
    "stop_on_end_of_data": True,        # end episode when we pass last row of the chosen instrument
    "strict_capital_check": True,       # disallow new positions if capital < (price + brokerage)
    "observation_window": 30,            # how many past bars to include in each observation
    "use_risk_adjusted_logging": True   # track approximate step returns for user to compute Sharpe, etc.
}

# -----------------------------------------------------------------------------------
# 2) ENVIRONMENT CELL
# -----------------------------------------------------------------------------------

import gym
import numpy as np
import pandas as pd
import random

class SingleInstrumentTradingEnv(gym.Env):
    """
    A robust environment to train on a SINGLE instrument at a time, with detailed logging.
    - If multiple instruments are provided in config["data_dict"], the environment will
      select one instrument (via config["instrument_selection"]) at reset(). This is useful
      for multi-task or MAML setups, but each episode focuses on one instrument only.
    - Capital is automatically set to config["initial_capital_multiplier"] * max(close)
      of the chosen instrument to ensure trades are feasible unless overwritten.
    - Drawdown penalty is included each step (reward -= alpha * current_drawdown).
    - End conditions:
        1) capital <= 0,
        2) drawdown >= max_drawdown_percent (if set),
        3) passing the last data bar if stop_on_end_of_data=True.
    - Step reward = (unrealized_reward + realized_reward - drawdown_penalty) * reward_scaling_factor.
    - We additionally log the "points_captured" or "points_lost" upon closing a position.
      (For a long close: exit_price - entry_price, for a short close: entry_price - exit_price)
    - This environment logs:
        * capital each step
        * win% each step
        * positions each step
        * approximate step returns for risk metrics if enabled
        * step-by-step logs (including points captured on close)
        * complete final evaluation report via get_evaluation_report().
    - Now includes a method get_step_logs_df() returning a DataFrame of all step logs.
    """

    def __init__(self, config):
        super(SingleInstrumentTradingEnv, self).__init__()

        # Store the config dict
        self.config = config

        # 1) Data-related
        self.data_dict = self.config["data_dict"]
        if not self.data_dict or len(self.data_dict) == 0:
            raise ValueError("config['data_dict'] must have at least one instrument DataFrame.")

        self.instrument_selection = self.config["instrument_selection"]

        self.current_instrument = None
        self.df = None
        self.max_steps = None

        # 2) Trading & Reward params
        self.initial_capital_multiplier = self.config["initial_capital_multiplier"]
        self.brokerage_per_side = self.config["brokerage_dict"][self._select_instrument()]
        self.flip_position = self.config["flip_position"]
        self.track_unrealized_in_reward = self.config["track_unrealized_in_reward"]
        self.unrealized_reward_weight = self.config["unrealized_reward_weight"]
        self.drawdown_penalty_alpha = self.config["drawdown_penalty_alpha"]

        self.reward_mode = self.config["reward_mode"]
        self.pct_base_capital = self.config["pct_base_capital"]
        self.clip_max_abs_reward = self.config["clip_max_abs_reward"]
        self.reward_scaling_factor = self.config["reward_scaling_factor"]

        self.max_drawdown_percent = self.config["max_drawdown_percent"]
        self.stop_on_end_of_data = self.config["stop_on_end_of_data"]
        self.strict_capital_check = self.config["strict_capital_check"]
        self.observation_window = self.config["observation_window"]
        self.use_risk_adjusted_logging = self.config["use_risk_adjusted_logging"]

        # 3) Internal trackers
        self.capital = 0.0
        self.initial_capital = 0.0  # set after we pick an instrument
        self.current_step = 0
        self.max_capital_seen = 0.0  # for drawdown
        self.position = 0           # 0=flat, 1=long, -1=short
        self.entry_price = 0.0
        self.unrealized_pnl = 0.0
        self.wins = 0
        self.losses = 0
        self.hold_step_count = 0    # how many steps we've held the current position
        self.trade_durations = []   # store each closed trade's hold steps

        # 4) Logs (step-by-step + aggregated)
        self.capital_log = []
        self.win_percentage_log = []
        self.position_type_log = []
        self.step_returns = []              # approximate step returns for risk metrics
        self.max_drawdown_logged = []

        # We'll store each step's info in a dictionary for detailed logs
        # including points_captured if a trade closes that step.
        self.step_logs = []  # each element => {"step":..., "capital":..., "win_percent":..., "position":..., ...}

        # 5) Spaces
        # Action space => (0=Hold, 1=Buy, 2=Sell)
        self.action_space = gym.spaces.Discrete(3)

        # We'll define observation_space shape dynamically after we pick an instrument in reset().
        self.observation_space = None

    def reset(self):
        """
        Resets the environment state, picks an instrument (if multiple),
        sets the capital, logs, positions, etc.
        """
        self.current_instrument = self._select_instrument()
        self.brokerage_per_side = self.config["brokerage_dict"][self.current_instrument]
        self.df = self.data_dict[self.current_instrument]
        self.max_steps = len(self.df)

        self.initial_capital = self._init_capital(self.df)
        self.capital = self.initial_capital
        self.max_capital_seen = self.capital

        self.current_step = 0
        self.position = 0
        self.entry_price = 0.0
        self.unrealized_pnl = 0.0
        self.wins = 0
        self.losses = 0
        self.hold_step_count = 0
        self.trade_durations.clear()

        # Clear logs
        self.capital_log.clear()
        self.win_percentage_log.clear()
        self.position_type_log.clear()
        self.step_returns.clear()
        self.max_drawdown_logged.clear()
        self.step_logs.clear()

        # Build observation space shape, now we know how many features & observation_window
        sample_obs = self._get_observation()
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=sample_obs.shape, dtype=np.float32
        )

        return sample_obs

    def step(self, action):
        """
        Execute one step:
          - Update unrealized PnL
          - Perform the action (Buy, Sell, Hold)
          - Calculate realized PnL if closing or flipping
          - Compute incremental reward from unrealized if track_unrealized_in_reward=True
          - Apply drawdown penalty
          - Log points captured or lost if a position is closed
          - Return (obs, reward, done, info)
        """
        done = False
        step_reward = 0.0

        if self.current_step >= self.max_steps:
            # No more data => done
            return self._get_observation(), 0.0, True, {}

        # 1) Compute current price & update unrealized
        row_idx = min(self.current_step, self.max_steps - 1)
        current_price = self.df['close'].iloc[row_idx]

        if self.position == 1:
            self.unrealized_pnl = current_price - self.entry_price
        elif self.position == -1:
            self.unrealized_pnl = self.entry_price - current_price
        else:
            self.unrealized_pnl = 0.0

        # If in a position, increment hold steps
        if self.position != 0:
            self.hold_step_count += 1
        else:
            self.hold_step_count = 0

        # Potential incremental reward from unrealized
        unreal_r = 0.0
        if self.track_unrealized_in_reward:
            unreal_r = self._transform_pnl_to_reward(self.unrealized_pnl)

        # We'll detect if a position gets closed or flipped and log the "points_captured".
        realized_pnl = 0.0
        points_captured = 0.0  # difference in price between entry and exit if we close

        # 2) Interpret action
        if self.position == 0:
            # Flat
            if action == 1:  # Buy
                if self._can_open_position(current_price):
                    self.position = 1
                    self.entry_price = current_price
                    self.capital -= self.brokerage_per_side
            elif action == 2:  # Sell
                if self._can_open_position(current_price):
                    self.position = -1
                    self.entry_price = current_price
                    self.capital -= self.brokerage_per_side

        elif self.position == 1:
            # Long
            if action == 2:
                # Close or flip
                realized_pnl = (current_price - self.entry_price)
                self.capital += realized_pnl
                self.capital -= self.brokerage_per_side

                # Points captured is (exit_price - entry_price) for a long
                points_captured = current_price - self.entry_price

                if realized_pnl > 0:
                    self.wins += 1
                else:
                    self.losses += 1

                self.trade_durations.append(self.hold_step_count)

                if self.flip_position:
                    # Flip to short
                    if self._can_open_position(current_price):
                        self.position = -1
                        self.entry_price = current_price
                        self.unrealized_pnl = 0.0
                        self.capital -= self.brokerage_per_side
                        self.hold_step_count = 1
                    else:
                        self.position = 0
                        self.entry_price = 0.0
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 0
                else:
                    # Flatten
                    self.position = 0
                    self.entry_price = 0.0
                    self.unrealized_pnl = 0.0
                    self.hold_step_count = 0

        else:
            # Short
            if action == 1:
                realized_pnl = (self.entry_price - current_price)
                self.capital += realized_pnl
                self.capital -= self.brokerage_per_side

                # Points captured for a short is (entry_price - exit_price)
                points_captured = self.entry_price - current_price

                if realized_pnl > 0:
                    self.wins += 1
                else:
                    self.losses += 1

                self.trade_durations.append(self.hold_step_count)

                if self.flip_position:
                    # Flip to long
                    if self._can_open_position(current_price):
                        self.position = 1
                        self.entry_price = current_price
                        self.unrealized_pnl = 0.0
                        self.capital -= self.brokerage_per_side
                        self.hold_step_count = 1
                    else:
                        self.position = 0
                        self.entry_price = 0.0
                        self.unrealized_pnl = 0.0
                        self.hold_step_count = 0
                else:
                    self.position = 0
                    self.entry_price = 0.0
                    self.unrealized_pnl = 0.0
                    self.hold_step_count = 0

        # 3) Realized reward
        realized_r = self._transform_pnl_to_reward(realized_pnl)

        # 4) Update capital peak, compute drawdown
        self.max_capital_seen = max(self.max_capital_seen, self.capital)
        current_drawdown = (self.max_capital_seen - self.capital) / max(self.max_capital_seen, 1e-9)

        # 5) Drawdown penalty
        drawdown_penalty = self.drawdown_penalty_alpha * current_drawdown
        drawdown_r = -drawdown_penalty

        # 6) Sum up step reward
        step_reward = ((unreal_r * self.unrealized_reward_weight) + realized_r + drawdown_r) * self.reward_scaling_factor

        # 7) Check end conditions
        if self.capital <= 0:
            done = True
        if (self.max_drawdown_percent is not None) and (current_drawdown >= self.max_drawdown_percent):
            done = True
        if self.stop_on_end_of_data and (self.current_step + 1 >= self.max_steps):
            done = True

        # 8) Logging
        self.capital_log.append(self.capital)
        total_trades = self.wins + self.losses
        if total_trades > 0:
            win_percent = (self.wins / total_trades) * 100.0
        else:
            win_percent = 0.0
        self.win_percentage_log.append(win_percent)

        if self.position == 1:
            pos_str = f"Long-{self.current_instrument}"
        elif self.position == -1:
            pos_str = f"Short-{self.current_instrument}"
        else:
            pos_str = "Flat"
        self.position_type_log.append(pos_str)
        self.max_drawdown_logged.append(current_drawdown)

        # track step returns for approximate Sharpe
        if self.use_risk_adjusted_logging:
            if len(self.capital_log) > 1:
                prev_cap = self.capital_log[-2]
                step_ret = (self.capital - prev_cap) / max(abs(prev_cap), 1e-9)
            else:
                step_ret = 0.0
            self.step_returns.append(step_ret)

        # store to self.step_logs for a complete step-by-step record
        # including points_captured if a position was closed/ flipped
        self.step_logs.append({
            "step": self.current_step,
            "capital": self.capital,
            "win_percent": win_percent,
            "position": pos_str,
            "unrealized_pnl": self.unrealized_pnl,
            "realized_pnl": realized_pnl,
            "points_captured": points_captured,  # only nonzero if a position closed/flipped this step
            "drawdown_fraction": current_drawdown,
            "reward": step_reward
        })

        # 9) Advance step and build next observation
        self.current_step += 1
        obs = self._get_observation()

        return obs, step_reward, done, {}

    def render(self, mode='human'):
        """Optional: Print debug info each step."""
        if len(self.capital_log) > 0:
            idx = len(self.capital_log) - 1
            cap_str = f"{self.capital_log[idx]:.2f}"
            winp_str = f"{self.win_percentage_log[idx]:.2f}"
            pos_str = self.position_type_log[idx]
            dd_str = f"{self.max_drawdown_logged[idx]*100:.2f}%"
            print(f"Step: {self.current_step}, "
                  f"Cap: {cap_str}, Win%: {winp_str}, Pos: {pos_str}, "
                  f"DD: {dd_str}")
        else:
            print(f"Step: {self.current_step}, Cap: {self.capital:.2f}, Pos: [No log yet]")

    def close(self):
        pass

    # -------------------------------------------------------------------
    # LOGGING & EVALUATION METHODS
    # -------------------------------------------------------------------
    def get_evaluation_report(self):
        """
        Returns a dictionary with final (or current) evaluation metrics:
          - 'instrument': which instrument was traded
          - 'initial_capital'
          - 'final_capital'
          - 'net_profit'
          - 'final_winrate'
          - 'capital_log': list of capital after each step
          - 'winrate_each_step': list of win % after each step
          - 'position_each_step': list of position states after each step
          - 'drawdown_each_step': list of drawdowns after each step
          - 'max_drawdown': final or max of drawdown_each_step
          - 'step_logs': detailed per-step records (including 'points_captured' on closes)
        """
        final_capital = self.capital_log[-1] if self.capital_log else self.capital
        net_profit = final_capital - self.initial_capital

        final_winrate = self.win_percentage_log[-1] if self.win_percentage_log else 0.0
        dd_list = self.max_drawdown_logged
        max_dd = max(dd_list) if len(dd_list) > 0 else 0.0

        report = {
            "instrument": self.current_instrument,
            "initial_capital": self.initial_capital,
            "final_capital": final_capital,
            "net_profit": net_profit,
            "final_winrate_percent": final_winrate,
            "capital_log": self.capital_log.copy(),
            "winrate_each_step": self.win_percentage_log.copy(),
            "position_each_step": self.position_type_log.copy(),
            "drawdown_each_step": dd_list.copy(),
            "max_drawdown_fraction": max_dd,
            "step_logs": self.step_logs.copy()
        }

        # Optional: approximate Sharpe ratio if step_returns are tracked
        if self.use_risk_adjusted_logging and len(self.step_returns) > 1:
            rets = np.array(self.step_returns)
            avg_r = rets.mean()
            std_r = rets.std()
            sharpe_approx = (avg_r / std_r) if std_r > 1e-9 else 0.0
            report["approx_sharpe"] = sharpe_approx

        return report

    def get_step_logs_df(self):
        """
        Convert the step_logs (list of dicts) into a pandas DataFrame for easier analysis.
        Each row corresponds to one environment step. Includes columns:
        ['step','capital','win_percent','position','unrealized_pnl','realized_pnl',
         'points_captured','drawdown_fraction','reward']
        """
        return pd.DataFrame(self.step_logs)

    # -------------------------------------------------------------------
    # INTERNAL HELPER METHODS
    # -------------------------------------------------------------------
    def _select_instrument(self):
        """
        Select an instrument for this episode.
        - "random": pick randomly from data_dict keys
        - "sequential": pick next in round-robin
        - otherwise: must be an exact key in data_dict
        """
        assets = list(self.data_dict.keys())

        if self.instrument_selection == "random":
            chosen = random.choice(assets)
        elif self.instrument_selection == "sequential":
            if not hasattr(self, '_seq_index'):
                self._seq_index = 0
            chosen = assets[self._seq_index % len(assets)]
            self._seq_index += 1
        else:
            # assume it's a direct string
            if self.instrument_selection not in assets:
                raise ValueError(f"instrument_selection='{self.instrument_selection}' not in data_dict.")
            chosen = self.instrument_selection

        return chosen

    def _init_capital(self, df):
        """
        Sets initial capital = initial_capital_multiplier * max(close).
        """
        max_close = df['close'].max()
        init_cap = self.initial_capital_multiplier * max_close
        return init_cap

    def _can_open_position(self, price):
        """
        If strict_capital_check=True, ensure we have enough capital to open with 'price'+brokerage.
        Otherwise, return True.
        """
        if not self.strict_capital_check:
            return True
        cost_to_open = price + self.brokerage_per_side
        return (self.capital >= cost_to_open)

    def _transform_pnl_to_reward(self, pnl):
        """
        Convert raw PnL => final reward (raw/pct/log/clip).
        """
        mode = self.reward_mode

        if mode == "raw":
            return pnl

        elif mode == "pct":
            if self.pct_base_capital == "current":
                base = max(self.capital, 1e-9)
            else:
                base = max(self.initial_capital, 1e-9)
            return pnl / base

        elif mode == "log":
            if self.pct_base_capital == "current":
                base = max(self.capital, 1e-9)
            else:
                base = max(self.initial_capital, 1e-9)
            ret = pnl / base
            ret_clamped = max(ret, -0.999999)  # avoid log(<=0)
            return np.log1p(ret_clamped)

        elif mode == "clip":
            clipped_val = np.clip(pnl, -self.clip_max_abs_reward, self.clip_max_abs_reward)
            return (clipped_val / self.clip_max_abs_reward) * 100.0

        return pnl  # fallback

    def _get_observation(self):
        """
        Gather observation by stacking the last `observation_window` rows
        plus capital, position, and unrealized PnL.
        If current_step < observation_window, replicate earliest row as needed.
        """
        obs_frames = []
        for offset in range(self.observation_window):
            step_idx = self.current_step - (self.observation_window - 1 - offset)
            step_idx = max(step_idx, 0)
            step_idx = min(step_idx, self.max_steps - 1)
            row_data = self.df.iloc[step_idx].values  # e.g. [open, high, low, close, ...]
            obs_frames.append(row_data)

        # Flatten stacked window
        obs_data = np.concatenate(obs_frames, axis=0).tolist()

        # Append capital
        obs_data.append(self.capital)

        # Append position & unrealized
        obs_data.append(float(self.position))
        obs_data.append(self.unrealized_pnl)

        return np.array(obs_data, dtype=np.float32)

In [15]:
# Start with low volatility instruments and gradually move to volatile instruments to allow the model to generalize better.
config["data_dict"] = {
    "Nifty": nf_processed_train_df,
    "BankNifty": bnf_processed_train_df
    # add more instruments if needed
}

config["brokerage_dict"] = {
    "Nifty": 20.0,      # ₹20 per trade
    "BankNifty": 20.0,  # ₹20 per trade
}

config["instrument_selection"] = "Nifty"  # "random" or "sequential" or "particular data name"

env = SingleInstrumentTradingEnv(config)
obs = env.reset()
done = False
while not done:
    action = env.action_space.sample()  # or from your RL policy
    obs, reward, done, info = env.step(action)
    env.render()


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Step: 1, Cap: 78800.35, Win%: 0.00, Pos: Short-Nifty, DD: 0.03%
Step: 2, Cap: 78760.05, Win%: 0.00, Pos: Long-Nifty, DD: 0.08%
Step: 3, Cap: 78760.05, Win%: 0.00, Pos: Long-Nifty, DD: 0.08%
Step: 4, Cap: 78760.05, Win%: 0.00, Pos: Long-Nifty, DD: 0.08%
Step: 5, Cap: 78760.05, Win%: 0.00, Pos: Long-Nifty, DD: 0.08%
Step: 6, Cap: 78760.05, Win%: 0.00, Pos: Long-Nifty, DD: 0.08%
Step: 7, Cap: 78760.05, Win%: 0.00, Pos: Long-Nifty, DD: 0.08%
Step: 8, Cap: 78718.35, Win%: 0.00, Pos: Short-Nifty, DD: 0.13%
Step: 9, Cap: 78718.35, Win%: 0.00, Pos: Short-Nifty, DD: 0.13%
Step: 10, Cap: 78680.25, Win%: 33.33, Pos: Long-Nifty, DD: 0.18%
Step: 11, Cap: 78680.25, Win%: 33.33, Pos: Long-Nifty, DD: 0.18%
Step: 12, Cap: 78633.65, Win%: 25.00, Pos: Short-Nifty, DD: 0.24%
Step: 13, Cap: 78633.65, Win%: 25.00, Pos: Short-Nifty, DD: 0.24%
Step: 14, Cap: 78633.65, Win%: 25.00, Pos: Short-Nifty, DD: 0.24%
Step: 15, Cap: 78633.65, Win%: 25.00, Pos: Short-Nifty, DD: 0.24%
Step: 16, Cap: 78605.35, Win%: 40.00

In [16]:
# After the episode, you can compute stats like approximate Sharpe ratio
if config["use_risk_adjusted_logging"]:
    rets = np.array(env.step_returns)
    avg_r = rets.mean()
    std_r = rets.std()
    sharpe_approx = avg_r / std_r if std_r > 1e-9 else 0
    print("Approx Sharpe:", sharpe_approx)

Approx Sharpe: -0.6880781485552524


In [17]:
report = env.get_evaluation_report()

print("Instrument:", report["instrument"])
print("Initial Capital:", report["initial_capital"])
print("Final Capital:", report["final_capital"])
print("Net Profit:", report["net_profit"])
print("Final Win Rate (%):", report["final_winrate_percent"])
print("Max Drawdown (%):", report["max_drawdown_fraction"] * 100.0)

Instrument: Nifty
Initial Capital: 78820.35
Final Capital: 39399.35000000011
Net Profit: -39420.9999999999
Final Win Rate (%): 51.271617497456766
Max Drawdown (%): 50.01373376291769


In [18]:
step_df = env.get_step_logs_df()
step_df

,step,capital,win_percent,position,unrealized_pnl,realized_pnl,points_captured,drawdown_fraction,reward
0,0,78800.35,0.000000,Short-Nifty,0.0,0.0,0.0,0.000254,-0.003806
1,1,78760.05,0.000000,Long-Nifty,0.0,-0.3,-0.3,0.000765,-0.011480
2,2,78760.05,0.000000,Long-Nifty,1.4,0.0,0.0,0.000765,-0.011470
3,3,78760.05,0.000000,Long-Nifty,-9.2,0.0,0.0,0.000765,-0.011510
4,4,78760.05,0.000000,Long-Nifty,-8.5,0.0,0.0,0.000765,-0.011508
...,...,...,...,...,...,...,...,...,...
2741,2741,39486.65,51.376147,Long-Nifty,7.2,0.0,0.0,0.499030,-7.485419
2742,2742,39486.65,51.376147,Long-Nifty,7.4,0.0,0.0,0.499030,-7.485418
2743,2743,39486.65,51.376147,Long-Nifty,1.3,0.0,0.0,0.499030,-7.485441
2744,2744,39444.65,51.323829,Short-Nifty,0.0,-2.0,-2.0,0.499563,-7.493472


MAML Agent

In [41]:
# ============================
# Cell 1: Imports and Config
# ============================
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from einops import rearrange
from collections import OrderedDict, deque

import higher  # We'll use 'higher' as an alternative to torchmeta for inner-loop adaptation

# ---------------------------
# 1. Configuration File (maml_config.py equivalent)
# ---------------------------

# Hardware
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TORCH_SEED = 42
torch.manual_seed(TORCH_SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(TORCH_SEED)

# Data Configuration
# Note: nf_processed_train_df, bnf_processed_train_df etc. are assumed to be defined above or elsewhere.
DATA_PATHS = OrderedDict({
    "NIFTY": nf_processed_train_df,
    "BANKNIFTY": bnf_processed_train_df
})

# Environment Parameters
ENV_CONFIG = {
    "initial_capital_multiplier": 3.0,
    "brokerage_dict": {
        "NIFTY": 20.0,
        "BANKNIFTY": 20.0
    },
    "observation_window": 30,
    "max_drawdown_percent": 0.5,
    "reward_mode": "pct",
    "pct_base_capital": "initial"
}

# Model Architecture
MODEL_CONFIG = {
    "input_dim": None,  # Set dynamically from environment
    "hidden_dim": 768,
    "num_heads": 8,
    "tcn_layers": 4,
    "transformer_layers": 3,
    "memory_size": 512,
    "regime_classes": 3
}

# Training Parameters
TRAIN_CONFIG = {
    "meta_lr": 3e-4,
    "adaptation_steps": 5,
    "meta_batch_size": 4,
    "max_epochs": 10000,
    "checkpoint_freq": 50,
    "entropy_coef": 0.01,
    "grad_clip": 1.0,
    # Curriculum for multi-instrument training
    "curriculum_stages": [
        ["USDGBP", "GOLD"],
        ["NIFTY", "BANKNIFTY"],
        ["BANKNIFTY", "USDGBP"]
    ]
}

# Online Learning
ONLINE_CONFIG = {
    "replay_size": 1000,
    "update_freq": 100,
    "live_window_size": 30,
    "checkpoint_dir": "./checkpoints"
}


In [35]:
# ============================
# Cell 2: Environment Placeholder / AntiOverfitMonitor
# ============================

# Simple AntiOverfitMonitor as a placeholder
class AntiOverfitMonitor:
    def __init__(self, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_value = -np.inf

    def check(self, train_loss, val_metric):
        """
        Simple logic: If the validation metric (e.g. Sharpe) hasn't improved
        in 'patience' checks, return True to trigger early stopping.
        """
        improved = val_metric > self.best_value
        if improved:
            self.best_value = val_metric
            self.counter = 0
        else:
            self.counter += 1

        return self.counter >= self.patience


In [36]:
# ============================
# Cell 3: Model Architectures (TCN & Transformer)
# ============================
class MarketAdaptiveTCN(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        self.hidden_dim = config['hidden_dim']
        self.tcn_layers = config['tcn_layers']
        self.layers = nn.ModuleList([
            nn.Conv1d(
                in_channels=input_dim if i == 0 else config['hidden_dim'],
                out_channels=config['hidden_dim'],
                kernel_size=3,
                dilation=2**i,
                padding=2**i
            )
            for i in range(config['tcn_layers'])
        ])

    def forward(self, x):
        # x shape: [Batch, SeqLen, InputDim]
        # Convert to [Batch, InputDim, SeqLen] for Conv1d
        x = x.permute(0, 2, 1)

        for layer in self.layers:
            x = torch.relu(layer(x))

        # Return shape: [Batch, SeqLen, HiddenDim]
        x = x.permute(0, 2, 1)
        return x


class CrossInstrumentTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_dim = config['hidden_dim']
        self.num_heads = config['num_heads']
        self.transformer_layers = config['transformer_layers']
        self.regime_classes = config['regime_classes']

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.hidden_dim,
            nhead=self.num_heads,
            dim_feedforward=self.hidden_dim * 4
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.transformer_layers)

        self.regime_predictor = nn.Linear(self.hidden_dim, self.regime_classes)

    def forward(self, x):
        # x shape: [Batch, SeqLen, HiddenDim]
        # TransformerEncoder expects [SeqLen, Batch, HiddenDim]
        x_trans = x.permute(1, 0, 2)
        encoded = self.encoder(x_trans)  # [SeqLen, Batch, HiddenDim]
        encoded = encoded.permute(1, 0, 2)  # [Batch, SeqLen, HiddenDim]

        # Regime classification from mean pooled features
        regime_logits = self.regime_predictor(encoded.mean(dim=1))  # [Batch, RegimeClasses]
        regime = torch.softmax(regime_logits, dim=-1)

        return encoded, regime


In [37]:
# ============================
# Cell 4: Main MAML Model (Replacing torchmeta's MetaModule with nn.Module)
# ============================
class MultiInstrumentMAML(nn.Module):
    def __init__(self, env, model_config):
        super().__init__()

        # Obtain input_dim from environment or user config
        self.input_dim = model_config['input_dim']
        if self.input_dim is None:
            # fallback to environment's observation_space size if not set
            self.input_dim = env.observation_space.shape[0]

        # Initialize submodules
        self.tcn = MarketAdaptiveTCN(self.input_dim, model_config)
        self.transformer = CrossInstrumentTransformer(model_config)

        self.memory_size = model_config['memory_size']
        self.hidden_dim = model_config['hidden_dim']

        # LSTM memory for task adaptation
        self.memory = nn.LSTMCell(self.hidden_dim, self.memory_size)

        # Projector for integrating external task embeddings if needed
        self.task_projector = nn.Linear(self.hidden_dim * 2, self.hidden_dim)

        # Policy network
        self.policy = nn.Sequential(
            nn.LayerNorm(self.memory_size),
            nn.Linear(self.memory_size, self.hidden_dim),
            nn.GELU(),
            nn.Linear(self.hidden_dim, 3)  # E.g. 3 actions: Hold / Buy / Sell
        )

        # Value network
        self.value = nn.Sequential(
            nn.LayerNorm(self.memory_size),
            nn.Linear(self.memory_size, self.hidden_dim),
            nn.GELU(),
            nn.Linear(self.hidden_dim, 1)
        )

        self.to(DEVICE)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, obs, hidden=None, task_emb=None):
        # obs shape: [Batch, SeqLen, InputDim]
        tcn_out = self.tcn(obs)  # [Batch, SeqLen, HiddenDim]
        trans_out, regime = self.transformer(tcn_out)  # [Batch, SeqLen, HiddenDim], [Batch, RegimeClasses]

        # Global context from mean pooling
        context = trans_out.mean(dim=1)  # [Batch, HiddenDim]

        # LSTM hidden/cell
        if hidden is None:
            batch_size = obs.size(0)
            h = torch.zeros(batch_size, self.memory_size, device=DEVICE)
            c = torch.zeros(batch_size, self.memory_size, device=DEVICE)
        else:
            (h, c) = hidden

        if task_emb is not None:
            # Condition on external task embedding
            context = torch.cat([context, task_emb], dim=1)
            context = self.task_projector(context)

        # LSTM memory update
        h, c = self.memory(context, (h, c))

        # Policy logits & value
        logits = self.policy(h)
        value = self.value(h)

        # Return the new hidden state for next step
        return logits, value, (h.detach(), c.detach()), regime


In [38]:
# ============================
# Cell 5: Trainer and Support Functions
# ============================
class HedgeFundTrainer:
    def __init__(self, env, model_config, train_config, data_paths):
        self.env = env
        self.agent = MultiInstrumentMAML(env, model_config).to(DEVICE)
        self.optimizer = optim.AdamW(self.agent.parameters(), lr=train_config['meta_lr'])

        self.scaler = torch.cuda.amp.GradScaler() if DEVICE == 'cuda' else None

        self.best_sharpe = -np.inf
        self.train_config = train_config
        self.data_paths = data_paths

        # For monitoring
        self._current_loss = 0.0
        self.overfit_monitor = AntiOverfitMonitor()

        # Pre-load data for all instruments
        self.task_data = self._load_all_instruments()

    def _load_all_instruments(self):
        """Load and preprocess all instruments as PyTorch Tensors."""
        task_dict = {}
        for instrument, data_source in self.data_paths.items():
            df = pd.read_parquet(data_source)
            trajectories = self._process_to_trajectories(df)
            split_idx = int(0.8 * len(trajectories))
            task_dict[instrument] = {
                'adapt': trajectories[:split_idx],
                'holdout': trajectories[split_idx:]
            }
        return task_dict

    def _process_to_trajectories(self, df, window=100):
        """Convert raw data to training trajectories of shape [window, features]."""
        obs = []
        mat = df.values
        for i in range(len(df) - window):
            obs_window = torch.FloatTensor(mat[i:i+window])
            obs.append(obs_window)
        return obs

    def _sample_task(self, stage=0):
        """Sample a single task from the curriculum stage."""
        instruments = self.train_config['curriculum_stages'][stage]
        instrument = random.choice(instruments)

        adapt_data = self.task_data[instrument]['adapt']
        holdout_data = self.task_data[instrument]['holdout']

        adapt_batch = random.sample(adapt_data, self.train_config['meta_batch_size'])
        holdout_batch = random.sample(holdout_data, self.train_config['meta_batch_size'])

        return {
            'adapt': torch.stack(adapt_batch).to(DEVICE),   # [batch_size, window, features]
            'holdout': torch.stack(holdout_batch).to(DEVICE),
            'instrument': instrument
        }

    def _calculate_returns(self, batch):
        """
        Placeholder for return calculation logic.
        This should match your environment's reward structure.
        For now, we just return zeros as a placeholder.
        """
        batch_size = batch.shape[0]
        return torch.zeros(batch_size, device=DEVICE)

    def _compute_loss(self, model, batch):
        """
        Compute total loss = Policy Loss + Entropy Regularization + Value Loss
        """
        logits, values, _, _ = model(batch)

        # Actions as "ground truth" => for demonstration we just label
        # them as the argmax of the logits themselves (self-supervised style).
        actions = torch.argmax(logits, dim=-1)

        # Policy loss (CrossEntropy)
        policy_loss = nn.CrossEntropyLoss()(logits, actions)

        # Entropy loss
        probs = torch.softmax(logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()

        # Value loss
        returns = self._calculate_returns(batch)
        value_loss = nn.MSELoss()(values.squeeze(), returns)

        # Weighted sum
        total_loss = policy_loss + self.train_config['entropy_coef'] * entropy + 0.5 * value_loss
        return total_loss

    def adapt(self, task_batch):
        """Fast adaptation on one task using 'higher' for the inner loop."""
        with higher.innerloop_ctx(self.agent, self.optimizer, copy_initial_weights=False) as (fnet, diffopt):
            # Adaptation steps
            for _ in range(self.train_config['adaptation_steps']):
                loss = self._compute_loss(fnet, task_batch['adapt'])
                diffopt.step(loss)

            # Evaluate on holdout set
            holdout_loss = self._compute_loss(fnet, task_batch['holdout'])
            holdout_loss.backward()

        return holdout_loss.item()

    def meta_update(self, meta_batch):
        """
        Aggregate updates across tasks.
        Here, we'll do a simple approach of collecting grads then applying them.
        You can also do advanced gradient manipulation if desired.
        """
        self.optimizer.zero_grad()
        task_gradients = []

        for task in meta_batch:
            loss_val = self.adapt(task)
            self._current_loss = loss_val  # store the latest loss

            # Collect gradients for each parameter
            grads = []
            for p in self.agent.parameters():
                # p.grad can be None if p has no gradient
                if p.grad is None:
                    grads.append(torch.zeros_like(p))
                else:
                    grads.append(p.grad.clone())

            # Reset gradient in preparation for the next task
            self.optimizer.zero_grad()
            task_gradients.append(grads)

        # Mean gradients across tasks
        mean_grads = []
        for grads_per_param in zip(*task_gradients):
            mean_grad = torch.stack(grads_per_param, dim=0).mean(dim=0)
            mean_grads.append(mean_grad)

        # Apply aggregated gradients
        for param, grad in zip(self.agent.parameters(), mean_grads):
            param.grad = grad

        # Gradient clipping
        nn.utils.clip_grad_norm_(self.agent.parameters(), self.train_config['grad_clip'])

        # Finally update
        self.optimizer.step()

    def train(self):
        """
        Full training loop with a simple curriculum:
        stage 0 => up to 0.33*max_epochs
        stage 1 => up to 0.66*max_epochs
        stage 2 => up to max_epochs
        """
        stage = 0

        for epoch in range(self.train_config['max_epochs']):
            # Determine stage
            if epoch > 0.33 * self.train_config['max_epochs']:
                stage = 1
            if epoch > 0.66 * self.train_config['max_epochs']:
                stage = 2

            # Sample a meta-batch of tasks
            meta_batch = [self._sample_task(stage) for _ in range(self.train_config['meta_batch_size'])]

            # Meta update
            self.meta_update(meta_batch)

            # Periodic checkpoint
            if epoch % self.train_config['checkpoint_freq'] == 0:
                val_sharpe = self.validate()

                if val_sharpe > self.best_sharpe:
                    self.best_sharpe = val_sharpe
                    self.save_checkpoint(epoch)

                # Early stopping check
                if self.overfit_monitor.check(self._current_loss, val_sharpe):
                    print(f"Early stopping at epoch {epoch}")
                    break

    def validate(self):
        """
        Cross-instrument validation.
        Here we just do a quick check on holdout sets and compute a pseudo Sharpe.
        """
        self.agent.eval()
        returns_list = []

        with torch.no_grad():
            for instrument in self.data_paths.keys():
                # Sample 8 holdout windows to estimate returns
                holdout_data = self.task_data[instrument]['holdout']
                if len(holdout_data) < 8:
                    continue
                batch = torch.stack(random.sample(holdout_data, 8)).to(DEVICE)
                _, values, _, _ = self.agent(batch)
                # For simplicity, treat these 'values' as pseudo returns
                returns_list.extend(values.cpu().numpy().flatten())

        if len(returns_list) == 0:
            # If no holdout data, return a dummy
            sharpe = -999
        else:
            ret_array = np.array(returns_list)
            sharpe = np.mean(ret_array) / (np.std(ret_array) + 1e-9)

        self.agent.train()
        return sharpe

    def save_checkpoint(self, epoch):
        """Save full training state"""
        if not os.path.exists(ONLINE_CONFIG['checkpoint_dir']):
            os.makedirs(ONLINE_CONFIG['checkpoint_dir'])

        path = os.path.join(ONLINE_CONFIG['checkpoint_dir'], f"maml_{epoch}.pth")
        torch.save({
            'epoch': epoch,
            'model': self.agent.state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'best_sharpe': self.best_sharpe,
            'task_data': self.task_data
        }, path)
        print(f"Checkpoint saved at {path}")


In [39]:
# ============================
# Cell 6: Online Learning & LiveTrader
# ============================
class LiveTrader:
    def __init__(self, env, model_path):
        self.env = env
        self.agent = self._load_model(model_path)
        self.optimizer = optim.SGD(self.agent.parameters(), lr=1e-3)
        self.replay = deque(maxlen=ONLINE_CONFIG['replay_size'])

    def _load_model(self, path):
        """Load trained meta-model"""
        checkpoint = torch.load(path, map_location=DEVICE)
        model = MultiInstrumentMAML(self.env, MODEL_CONFIG)
        model.load_state_dict(checkpoint['model'])
        return model.to(DEVICE)

    def _calculate_returns(self, batch):
        # Similar to the trainer's approach
        batch_size = batch.shape[0]
        return torch.zeros(batch_size, device=DEVICE)

    def _compute_loss(self, model, batch):
        logits, values, _, _ = model(batch)
        actions = torch.argmax(logits, dim=-1)

        # Policy loss
        policy_loss = nn.CrossEntropyLoss()(logits, actions)

        # Entropy
        probs = torch.softmax(logits, dim=-1)
        entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()

        # Value loss
        returns = self._calculate_returns(batch)
        value_loss = nn.MSELoss()(values.squeeze(), returns)

        total_loss = policy_loss + 0.01 * entropy + 0.5 * value_loss
        return total_loss

    def process_live_data(self, data):
        """
        Convert live feed to the correct shape: [1, SeqLen, InputDim].
        'data' might already be an array of shape (window_size, features).
        """
        obs = torch.FloatTensor(data).to(DEVICE)
        # Expand dims to form a batch of size 1
        return obs.unsqueeze(0)

    def update_policy(self, batch):
        """Continual adaptation using 'higher' for the online setting."""
        self.agent.train()
        with higher.innerloop_ctx(self.agent, self.optimizer, track_higher_grads=False) as (fnet, diffopt):
            loss = self._compute_loss(fnet, batch)
            diffopt.step(loss)
        return loss.item()

    def trade_cycle(self):
        """
        Real-time trading loop.
        In practice, you'd break out of this loop based on market hours or signals.
        """
        while True:
            try:
                # 1. Get new live data
                data = self.env.get_live_window(ONLINE_CONFIG['live_window_size'])
                obs = self.process_live_data(data)
                self.replay.append(obs)

                # 2. Periodic update
                if len(self.replay) >= ONLINE_CONFIG['meta_batch_size'] and \
                   len(self.replay) % ONLINE_CONFIG['update_freq'] == 0:

                    batch = random.sample(self.replay, ONLINE_CONFIG['meta_batch_size'])
                    batch = torch.cat(batch, dim=0)
                    loss_val = self.update_policy(batch)
                    print(f"Online update loss: {loss_val}")

                # 3. Periodic checkpoint
                if int(time.time()) % 3600 == 0:  # e.g. every hour
                    self.save_state()

            except Exception as e:
                print(f"Trading error: {e}")
                self.save_state()
                time.sleep(60)  # Wait a bit before retry

    def save_state(self):
        """Save the online-updated model state (not the entire trainer)."""
        if not os.path.exists(ONLINE_CONFIG['checkpoint_dir']):
            os.makedirs(ONLINE_CONFIG['checkpoint_dir'])

        timestamp = int(time.time())
        path = os.path.join(ONLINE_CONFIG['checkpoint_dir'], f"live_maml_{timestamp}.pth")
        torch.save({
            'model': self.agent.state_dict(),
            'optimizer': self.optimizer.state_dict()
        }, path)
        print(f"Online checkpoint saved at {path}")


In [43]:
# ============================
# Cell 7: Execution Entry Point
# ============================
if __name__ == "__main__":
    # 1. Initialize environment
    env_config = ENV_CONFIG.copy()
    # Just in case you want to pass the dataframes directly to your environment
    # env_config["data_dict"] = {k: pd.read_parquet(v) for k, v in DATA_PATHS.items()}

    # We'll set MODEL_CONFIG["input_dim"] here if not set:
    # MODEL_CONFIG["input_dim"] = <some_value_from_data>

    env = SingleInstrumentTradingEnv(config)

    # 2. Train the MAML agent
    trainer = HedgeFundTrainer(env, MODEL_CONFIG, TRAIN_CONFIG, DATA_PATHS)
    trainer.train()

    # 3. Online trading (after training finishes)
    best_ckpt_path = os.path.join(ONLINE_CONFIG['checkpoint_dir'], "maml_best.pth")
    if os.path.exists(best_ckpt_path):
        trader = LiveTrader(env, best_ckpt_path)
        trader.trade_cycle()
    else:
        print("No trained checkpoint found. Please train and save the model first.")


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
<ipython-input-38-983f79d27ea0>:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler() if DEVICE == 'cuda' else None


TypeError: cannot construct a FileSource from                          open      high       low     close  rsi_base  macd  \
datetime                                                                      
2023-06-06 10:21:00  18576.20  18577.80  18573.70  18575.00     35.60 -7.67   
2023-06-06 10:23:00  18574.80  18576.30  18573.70  18575.30     36.09 -7.52   
2023-06-06 10:25:00  18575.90  18577.60  18574.10  18576.70     38.44 -7.21   
2023-06-06 10:27:00  18575.90  18576.40  18565.30  18566.10     29.58 -7.73   
2023-06-06 10:29:00  18566.40  18568.80  18565.70  18566.80     30.71 -7.99   
...                       ...       ...       ...       ...       ...   ...   
2024-10-18 15:21:00  24843.65  24856.95  24843.40  24855.75     48.83 -5.78   
2024-10-18 15:23:00  24857.10  24876.40  24855.65  24875.15     59.75 -3.46   
2024-10-18 15:25:00  24874.55  24876.35  24865.80  24866.40     54.14 -2.30   
2024-10-18 15:27:00  24868.40  24874.30  24863.20  24869.15     55.55 -1.15   
2024-1

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15, market_close_hour=15, market_close_minute=0):
    # Get current time in IST
    now = datetime.now(pytz.utc).astimezone(ist_timezone)

    # Define the market start and close times in IST for today
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)
    market_close_time = now.replace(hour=market_close_hour, minute=market_close_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before market starts, set next_run_time to market start time
        next_run_time = market_start_time
    elif now > market_close_time:
        # If current time is after market close, calculate the time until the next market open
        next_market_start_time = market_start_time + timedelta(days=1)
        next_run_time = next_market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": nf_index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(action, actual_closing_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()
    print(final_current_time)

    # Ensure trading occurs only during market hours
    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(18, 0):
        if action == 1 and not active_order:  # Buy action
            print("Buy action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making CE Position at {actual_closing_price}")
            ce_entry()  # Execute CE entry logic
        elif action == 2 and not active_order:  # Sell action
            print("Sell action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making PE Position at {actual_closing_price}")
            pe_entry()  # Execute PE entry logic
        elif action == 0:
            print("Hold action detected by PPO model. No trade executed.")

In [ ]:
def fetch_and_prepare_final_data():
    final_df = fetch_candle_data(10)

    final_data_processor = DataProcessor(final_df, live_processing=True)

    final_processed_df = final_data_processor.process().df

    return final_processed_df

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['atr'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [ ]:
# Online training configuration
online_learning_steps = 1000  # Steps for each retraining
retrain_interval_minutes = 60  # Retrain the model every 60 minutes

start_time = datetime.now()

while True:
    clear_output(wait=True)

    num_candles = 100

    final_df = fetch_and_prepare_final_data()

    final_df = final_df.iloc[:-1]

    target = final_df['Target'].iloc[-1]
    trailing_sl = final_df['Stop Loss'].iloc[-1]

    # Initialize the TradingEnvironment with the latest data
    live_env = create_env(final_df, config)
    obs = live_env.reset()

    # Use PPO model to predict action for the latest data point
    action, _ = ppo_model.predict(obs, deterministic=True)

    # Extract the actual closing price
    actual_closing_price = final_df['close'].iloc[-1]

    # Retrain PPO model at specified intervals
    if (datetime.now() - start_time).seconds >= retrain_interval_minutes * 60:
        print("Retraining PPO model with online data...")
        train_env = TradingEnvironment(final_df, config)  # Create a new training environment
        ppo_model.set_env(train_env)  # Update the PPO model's environment
        ppo_model.learn(total_timesteps=online_learning_steps)  # Retrain the model
        ppo_model.save(model_save_path)  # Save the updated model
        start_time = datetime.now()

    # Identify most recent high and low points
    recent_highs, recent_lows = find_local_extrema(final_df)

    most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
    most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

    high_trendline = [np.nan] * len(final_df)
    low_trendline = [np.nan] * len(final_df)

    if most_recent_high is not None:
        previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
        high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

    if most_recent_low is not None:
        previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
        low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

    # Prepare candlestick data for mplfinance
    actual_candles = final_df[-num_candles:].copy()

    # Create a DataFrame for mplfinance
    mpf_df = actual_candles[['open', 'high', 'low', 'close']]

    # Create addplot elements for predicted prices and actual close prices
    ap = [
        mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
        #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
    ]

    # Add trendlines to the plot
    if most_recent_high is not None:
        ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    if most_recent_low is not None:
        ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

    for ax in axlist:
        ax.grid(False)

    axlist[0].legend()
    plt.show()

    # Execute market entry/exit logic
    market_entry_exit_logic(action, actual_closing_price, final_df)

    sleep_time = get_sleep_time(interval_minutes)
    print(f"Sleeping for {sleep_time}")
    time.sleep(sleep_time)